In [2]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from datetime import datetime

In [3]:
# Step 1: Fetch the website
url = 'https://understat.com/league/EPL'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

In [4]:
# Step 2: Find the specific <script> tag containing the JSON data
scripts = soup.find_all('script')
json_data = None

In [5]:
# Step 3: Look for the script containing the teamsData
for script in scripts:
    if 'teamsData' in script.text:
        # Extract the content of the script tag as text
        json_text = script.text
        # Step 4: Isolate the JSON data by splitting the string
        json_text = json_text.split("JSON.parse('")[1].split("')")[0]
        # Step 5: Decode the string by replacing escape characters
        json_text = json_text.encode('utf-8').decode('unicode_escape')
        # Step 6: Load the decoded text as a Python dictionary
        json_data = json.loads(json_text)
        break

In [6]:
json_data

{'71': {'id': '71',
  'title': 'Aston Villa',
  'history': [{'h_a': 'a',
    'xG': 2.6703,
    'xGA': 2.8047,
    'npxG': 2.6703,
    'npxGA': 2.04353,
    'ppda': {'att': 260, 'def': 23},
    'ppda_allowed': {'att': 224, 'def': 24},
    'deep': 8,
    'deep_allowed': 5,
    'scored': 2,
    'missed': 1,
    'xpts': 1.2989000000000002,
    'result': 'w',
    'date': '2024-08-17 16:30:00',
    'wins': 1,
    'draws': 0,
    'loses': 0,
    'pts': 3,
    'npxGD': 0.62677},
   {'h_a': 'h',
    'xG': 1.31664,
    'xGA': 1.41399,
    'npxG': 1.31664,
    'npxGA': 1.41399,
    'ppda': {'att': 311, 'def': 18},
    'ppda_allowed': {'att': 175, 'def': 30},
    'deep': 3,
    'deep_allowed': 10,
    'scored': 0,
    'missed': 2,
    'xpts': 1.2362000000000002,
    'result': 'l',
    'date': '2024-08-24 16:30:00',
    'wins': 0,
    'draws': 0,
    'loses': 1,
    'pts': 0,
    'npxGD': -0.09735000000000005},
   {'h_a': 'a',
    'xG': 2.16385,
    'xGA': 0.774741,
    'npxG': 2.16385,
    'npxGA'

In [7]:
# Step 7: Extract match data for each team and identify the opponent
data = []

# Step 8: Create a reverse lookup dictionary for matches based on date and home/away status
matches_by_date = {}

for team_id, team_info in json_data.items():
    team_name = team_info['title']
    for match in team_info['history']:
        if match['h_a'] == 'h':
            match_key = (match['date'], match['xG'], match['xGA'])
        else:
            match_key = (match['date'], match['xGA'], match['xG'])

        if match_key not in matches_by_date:
            matches_by_date[match_key] = []
        matches_by_date[match_key].append({
            'team': team_name,
            'xG': match['xG'],
            'xGA': match['xGA'],
            'scored': match['scored'],
            'missed': match['missed'],
            'result': match['result'],
            'pts': match['pts'],
            'h_a': match['h_a']  # home/away status
        })

In [8]:
# Step 9: Construct final data with opponents
for match_key, matches in matches_by_date.items():
    if len(matches) == 2:
        # We found both home and away team for the match
        home_match = [m for m in matches if m['h_a'] == 'h'][0]
        away_match = [m for m in matches if m['h_a'] == 'a'][0]
        
        # Add home team data with away team as opponent
        data.append({
            'team': home_match['team'],
            'opponent': away_match['team'],
            'date': match_key[0],
            'xG': home_match['xG'],
            'xGA': home_match['xGA'],
            'scored': home_match['scored'],
            'missed': home_match['missed'],
            'result': home_match['result'],
            'pts': home_match['pts']
        })
        
        # Add away team data with home team as opponent
        data.append({
            'team': away_match['team'],
            'opponent': home_match['team'],
            'date': match_key[0],
            'xG': away_match['xG'],
            'xGA': away_match['xGA'],
            'scored': away_match['scored'],
            'missed': away_match['missed'],
            'result': away_match['result'],
            'pts': away_match['pts']
        })

In [9]:
# Step 10: Convert the list of match data into a pandas DataFrame
df = pd.DataFrame(data)
df

,team,opponent,date,xG,xGA,scored,missed,result,pts
0,West Ham,Aston Villa,2024-08-17 16:30:00,2.804700,2.670300,1,2,l,0
1,Aston Villa,West Ham,2024-08-17 16:30:00,2.670300,2.804700,2,1,w,3
2,Aston Villa,Arsenal,2024-08-24 16:30:00,1.316640,1.413990,0,2,l,0
3,Arsenal,Aston Villa,2024-08-24 16:30:00,1.413990,1.316640,2,0,w,3
4,Leicester,Aston Villa,2024-08-31 14:00:00,0.774741,2.163850,1,2,l,0
...,...,...,...,...,...,...,...,...,...
471,Ipswich,Brentford,2024-10-26 14:00:00,1.656980,4.096630,3,4,l,0
472,Brentford,Nottingham Forest,2024-12-21 15:00:00,0.769004,0.964897,0,2,l,0
473,Nottingham Forest,Brentford,2024-12-21 15:00:00,0.964897,0.769004,2,0,w,3
474,Nottingham Forest,Ipswich,2024-11-30 15:00:00,1.765870,0.462771,1,0,w,3


In [10]:
df[df['date'] > datetime.now().strftime("%Y-%m-%d %H:%M:%S")]

,team,opponent,date,xG,xGA,scored,missed,result,pts


In [11]:
df['date'].max()

'2025-02-02 16:30:00'

## Dates Data

In [12]:
# Step 3: Look for the script containing the teamsData
for script in scripts:
    if 'datesData' in script.text:
        # Extract the content of the script tag as text
        dates_json_text = script.text
        # Step 4: Isolate the JSON data by splitting the string
        dates_json_text = dates_json_text.split("JSON.parse('")[1].split("')")[0]
        # Step 5: Decode the string by replacing escape characters
        dates_json_text = dates_json_text.encode('utf-8').decode('unicode_escape')
        # Step 6: Load the decoded text as a Python dictionary
        dates_json_data = json.loads(dates_json_text)
        break

In [13]:
dates_json_data

[{'id': '26602',
  'isResult': True,
  'h': {'id': '89', 'title': 'Manchester United', 'short_title': 'MUN'},
  'a': {'id': '228', 'title': 'Fulham', 'short_title': 'FLH'},
  'goals': {'h': '1', 'a': '0'},
  'xG': {'h': '2.04268', 'a': '0.418711'},
  'datetime': '2024-08-16 19:00:00',
  'forecast': {'w': '0.8069', 'd': '0.1489', 'l': '0.0442'}},
 {'id': '26603',
  'isResult': True,
  'h': {'id': '285', 'title': 'Ipswich', 'short_title': 'IPS'},
  'a': {'id': '87', 'title': 'Liverpool', 'short_title': 'LIV'},
  'goals': {'h': '0', 'a': '2'},
  'xG': {'h': '0.342601', 'a': '3.92906'},
  'datetime': '2024-08-17 11:30:00',
  'forecast': {'w': '0.0026', 'd': '0.018', 'l': '0.9794'}},
 {'id': '26604',
  'isResult': True,
  'h': {'id': '83', 'title': 'Arsenal', 'short_title': 'ARS'},
  'a': {'id': '229', 'title': 'Wolverhampton Wanderers', 'short_title': 'WOL'},
  'goals': {'h': '2', 'a': '0'},
  'xG': {'h': '1.6283', 'a': '0.575835'},
  'datetime': '2024-08-17 14:00:00',
  'forecast': {'w': 

In [14]:
matches_by_date = {
    'match_id':[],
    'home_team_id':[],
    'home_team_long_title':[],
    'home_team_short_title':[],
    'away_team_id':[],
    'away_team_long_title':[],
    'away_team_short_title':[],
    'home_goals':[],
    'away_goals':[],
    'home_xg':[],
    'away_xg':[],
    'datetime':[],
}

for match in dates_json_data:
    matches_by_date['match_id'].append(match['id'])
    matches_by_date['home_team_id'].append(match['h']['id'])
    matches_by_date['home_team_long_title'].append(match['h']['title'])
    matches_by_date['home_team_short_title'].append(match['h']['short_title'])
    matches_by_date['away_team_id'].append(match['a']['id'])
    matches_by_date['away_team_long_title'].append(match['a']['title'])
    matches_by_date['away_team_short_title'].append(match['a']['short_title'])
    matches_by_date['home_goals'].append(match['goals']['h'])
    matches_by_date['away_goals'].append(match['goals']['a'])
    matches_by_date['home_xg'].append(match['xG']['h'])
    matches_by_date['away_xg'].append(match['xG']['a'])
    matches_by_date['datetime'].append(match['datetime'])

df_dates = pd.DataFrame(matches_by_date)

In [15]:
df_dates.head()

,match_id,home_team_id,home_team_long_title,home_team_short_title,away_team_id,away_team_long_title,away_team_short_title,home_goals,away_goals,home_xg,away_xg,datetime
0,26602,89,Manchester United,MUN,228,Fulham,FLH,1,0,2.04268,0.418711,2024-08-16 19:00:00
1,26603,285,Ipswich,IPS,87,Liverpool,LIV,0,2,0.342601,3.92906,2024-08-17 11:30:00
2,26604,83,Arsenal,ARS,229,Wolverhampton Wanderers,WOL,2,0,1.6283,0.575835,2024-08-17 14:00:00
3,26605,72,Everton,EVE,220,Brighton,BRI,0,3,0.405325,1.79083,2024-08-17 14:00:00
4,26606,86,Newcastle United,NEW,74,Southampton,SOU,1,0,0.433489,1.95483,2024-08-17 14:00:00


In [16]:
df_dates['datetime'].min()

'2024-08-16 19:00:00'

In [17]:
df_dates['datetime'].max()

'2025-05-25 16:00:00'

In [18]:
def load_understat_data():
    # Step 1: Fetch the website
    url = 'https://understat.com/league/EPL'
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Step 2: Find the specific <script> tag containing the JSON data
    scripts = soup.find_all('script')
    dates_json_data = None

    # Step 3: Look for the script containing the teamsData
    for script in scripts:
        if 'datesData' in script.text:
            # Extract the content of the script tag as text
            dates_json_text = script.text
            # Step 4: Isolate the JSON data by splitting the string
            dates_json_text = dates_json_text.split("JSON.parse('")[1].split("')")[0]
            # Step 5: Decode the string by replacing escape characters
            dates_json_text = dates_json_text.encode('utf-8').decode('unicode_escape')
            # Step 6: Load the decoded text as a Python dictionary
            dates_json_data = json.loads(dates_json_text)
            break

    if dates_json_data:
        matches_by_date = {
            'match_id':[],
            'home_team_id':[],
            'home_team_long_title':[],
            'home_team_short_title':[],
            'away_team_id':[],
            'away_team_long_title':[],
            'away_team_short_title':[],
            'home_goals':[],
            'away_goals':[],
            'home_xg':[],
            'away_xg':[],
            'datetime':[],
        }

        for match in dates_json_data:
            matches_by_date['match_id'].append(match['id'])
            matches_by_date['home_team_id'].append(match['h']['id'])
            matches_by_date['home_team_long_title'].append(match['h']['title'])
            matches_by_date['home_team_short_title'].append(match['h']['short_title'])
            matches_by_date['away_team_id'].append(match['a']['id'])
            matches_by_date['away_team_long_title'].append(match['a']['title'])
            matches_by_date['away_team_short_title'].append(match['a']['short_title'])
            matches_by_date['home_goals'].append(match['goals']['h'])
            matches_by_date['away_goals'].append(match['goals']['a'])
            matches_by_date['home_xg'].append(match['xG']['h'])
            matches_by_date['away_xg'].append(match['xG']['a'])
            matches_by_date['datetime'].append(match['datetime'])

        df_dates = pd.DataFrame(matches_by_date)
        return df_dates
    else:
        print("Data could not be found")